# 01 — Observable data-quality audit

This is a source audit, not modelling and not fault evaluation. It reads only
the observable synthetic PON panel, topology and optional service windows.
Tickets, injected faults and label files are intentionally outside this
notebook.

Outputs are compact evidence tables. The raw telemetry remains external to Git.


## 1. Setup and source inventory


In [ ]:
from pathlib import Path
import os
import sys


if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import shutil
import tempfile

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from telco_anomaly.adapters.synthetic_pon import inspect_synthetic_pon
from telco_anomaly.contract import truth_like_columns
from telco_anomaly.io import (
    immutable_output_directory, load_config, read_json, resolve_data_root,
    resolve_dataset_source, write_json,
)

DATA_ROOT = resolve_data_root()
SOURCE = resolve_dataset_source(
    "synthetic_pon", data_root=DATA_ROOT, project_root=PROJECT_ROOT
)
METRIC_CONFIG = load_config("metric_registry", project_root=PROJECT_ROOT)
RUN_ID = os.getenv("PON_AUDIT_RUN_ID", "synthetic_pon_source_v2")
OUTPUT_ROOT = DATA_ROOT / "audits" / "synthetic_pon" / RUN_ID

inventory = inspect_synthetic_pon(SOURCE, METRIC_CONFIG)
BASE_CADENCE_SECONDS = float(pd.DataFrame(METRIC_CONFIG["metrics"])["expected_cadence_seconds"].mode().iloc[0])
PANEL = inventory["panel"]
TOPOLOGY = inventory["topology"]

display(pd.Series({
    "source": str(SOURCE),
    "panel": PANEL.name,
    "topology": TOPOLOGY.name,
    "output": str(OUTPUT_ROOT),
    "native_columns": len(inventory["native_columns"]),
}, name="value").to_frame())


## 2. Classify every source column

Only declared measurements proceed to canonical telemetry. Identifiers and
device/topology attributes remain metadata. Generator-derived expectations and
business weights are explicitly excluded from model input. A new, unreviewed
column stops the audit instead of being silently accepted.


In [ ]:
declared_metrics = set(inventory["mapping"]["native_field"])
identity_fields = {"timestamp_utc", "ont_id"}
metadata_fields = {
    "olt_id", "pon_port", "splitter_l1", "splitter_l2", "geo_cluster",
    "vendor", "device_model", "enclosure", "firmware_version",
    "distance_m", "distance_bucket", "splitter_ratio",
    "l2_splitter_capacity", "fibre_age_yr",
}
excluded_generator_fields = {
    "expected_rx_power_dbm", "rx_sensitivity_dbm",
    "service_impact_weight", "customer_priority_weight",
}


def classify_column(name):
    if name in declared_metrics:
        return "safe_measurement"
    if name in identity_fields:
        return "identity_or_time"
    if name in metadata_fields:
        return "metadata_not_telemetry"
    if name in excluded_generator_fields:
        return "derived_or_business_field_excluded"
    if name in truth_like_columns([name]):
        return "evaluation_truth_rejected"
    return "unreviewed_rejected"


column_classification = pd.DataFrame({
    "column": inventory["native_columns"],
})
column_classification["classification"] = column_classification["column"].map(
    classify_column
)
display(column_classification)

unreviewed = column_classification.loc[
    column_classification["classification"].eq("unreviewed_rejected"), "column"
]
if len(unreviewed):
    raise ValueError(f"Review new source columns before continuing: {unreviewed.tolist()}")
assert not column_classification["classification"].str.contains("truth").any()


## 3. Shape, time coverage and key integrity


In [ ]:
panel_sql = str(PANEL).replace("'", "''")
connection = duckdb.connect()
connection.execute("SET threads = 2")
connection.execute("SET preserve_insertion_order = false")

overview = connection.execute(f"""
    WITH source AS (
        SELECT CAST(timestamp_utc AS TIMESTAMPTZ) AS event_ts,
               CAST(ont_id AS VARCHAR) AS entity_id
        FROM read_parquet('{panel_sql}')
    ), duplicate_keys AS (
        SELECT count(*) - count(DISTINCT (entity_id, event_ts)) AS duplicates
        FROM source
    )
    SELECT count(*) AS rows,
           count(DISTINCT entity_id) AS entities,
           min(event_ts) AS observed_from,
           max(event_ts) AS observed_to,
           (SELECT duplicates FROM duplicate_keys) AS duplicate_entity_timestamps
    FROM source
""").df()
display(overview.T.rename(columns={0: "value"}))
assert int(overview.loc[0, "duplicate_entity_timestamps"]) == 0


## 4. Entity coverage and observed cadence

The cadence check uses timestamps, not non-null values. A row present with a
null metric is a failed/invalid measurement; a missing timestamp is a
collection gap.


In [ ]:
entity_coverage = connection.execute(f"""
    WITH ordered AS (
        SELECT CAST(ont_id AS VARCHAR) AS entity_id,
               CAST(timestamp_utc AS TIMESTAMPTZ) AS event_ts,
               lag(CAST(timestamp_utc AS TIMESTAMPTZ)) OVER (
                   PARTITION BY ont_id ORDER BY timestamp_utc
               ) AS previous_ts
        FROM read_parquet('{panel_sql}')
    )
    SELECT entity_id,
           min(event_ts) AS observed_from,
           max(event_ts) AS observed_to,
           count(*) AS observed_rows,
           median(epoch(event_ts - previous_ts)) AS median_cadence_seconds,
           quantile_cont(epoch(event_ts - previous_ts), 0.95) AS p95_cadence_seconds,
           sum(epoch(event_ts - previous_ts) > {BASE_CADENCE_SECONDS} * 1.5) AS timestamp_gaps
    FROM ordered
    GROUP BY entity_id
    ORDER BY entity_id
""").df()
display(entity_coverage.describe(include="all").T)
assert entity_coverage["median_cadence_seconds"].dropna().eq(BASE_CADENCE_SECONDS).all()


## 5. Metric validity, tails, zero inflation and clipping


In [ ]:
# Build one wide aggregation so the large Parquet panel is scanned once.
expressions = ["count(*) AS rows"]
for item in METRIC_CONFIG["metrics"]:
    field = item["native_field"]
    clip_at = item.get("source_quality", {}).get("clipped_at")
    expressions.extend([
        f'count("{field}") AS "{field}__non_null"',
        f'sum(isfinite("{field}")) AS "{field}__finite"',
        f'sum("{field}" = 0) AS "{field}__zero"',
        (
            f'sum("{field}" >= {float(clip_at)}) AS "{field}__clipped"'
            if clip_at is not None else f'0 AS "{field}__clipped"'
        ),
        f'min("{field}") FILTER (WHERE isfinite("{field}")) AS "{field}__minimum"',
        f'approx_quantile("{field}", 0.05) FILTER (WHERE isfinite("{field}")) AS "{field}__q05"',
        f'approx_quantile("{field}", 0.50) FILTER (WHERE isfinite("{field}")) AS "{field}__median"',
        f'approx_quantile("{field}", 0.95) FILTER (WHERE isfinite("{field}")) AS "{field}__q95"',
        f'max("{field}") FILTER (WHERE isfinite("{field}")) AS "{field}__maximum"',
    ])

wide_quality = connection.execute(
    f"SELECT {', '.join(expressions)} FROM read_parquet('{panel_sql}')"
).df().iloc[0]

metric_rows = []
for item in METRIC_CONFIG["metrics"]:
    field = item["native_field"]
    metric_rows.append({
        "native_field": field,
        "metric_id": item["metric_id"],
        "measurement_kind": item["measurement_kind"],
        "rows": wide_quality["rows"],
        **{
            name: wide_quality[f"{field}__{name}"]
            for name in (
                "non_null", "finite", "zero", "clipped", "minimum",
                "q05", "median", "q95", "maximum",
            )
        },
    })

metric_quality_report = pd.DataFrame(metric_rows)
metric_quality_report["null_rate"] = 1 - (
    metric_quality_report["non_null"] / metric_quality_report["rows"]
)
metric_quality_report["nonfinite_rate"] = (
    metric_quality_report["non_null"] - metric_quality_report["finite"]
) / metric_quality_report["rows"]
metric_quality_report["zero_rate"] = (
    metric_quality_report["zero"] / metric_quality_report["finite"]
)
metric_quality_report["clipped_rate"] = (
    metric_quality_report["clipped"] / metric_quality_report["finite"]
)
metric_quality_report = metric_quality_report[[
    "native_field", "metric_id", "measurement_kind", "rows", "finite",
    "null_rate", "nonfinite_rate", "zero_rate", "clipped_rate",
    "minimum", "q05", "median", "q95", "maximum",
]]
display(metric_quality_report)


## 6. Counter-reset audit


In [ ]:
counter_rows = []
counter_fields = [
    item["native_field"] for item in METRIC_CONFIG["metrics"]
    if item["measurement_kind"] == "cumulative_counter"
]
for field in counter_fields:
    row = connection.execute(f"""
        WITH ordered AS (
            SELECT CAST(ont_id AS VARCHAR) AS entity_id,
                   CAST(timestamp_utc AS TIMESTAMPTZ) AS event_ts,
                   "{field}" AS value,
                   lag("{field}") OVER (
                       PARTITION BY ont_id ORDER BY timestamp_utc
                   ) AS previous_value
            FROM read_parquet('{panel_sql}')
        )
        SELECT '{field}' AS native_field,
               sum(value < previous_value) AS negative_increments,
               sum(value IS NOT NULL AND previous_value IS NOT NULL) AS comparable_pairs
        FROM ordered
    """).df().iloc[0].to_dict()
    counter_rows.append(row)

counter_audit = pd.DataFrame(counter_rows)
if not counter_audit.empty:
    counter_audit["negative_increment_rate"] = (
        counter_audit["negative_increments"] / counter_audit["comparable_pairs"]
    )
display(counter_audit)


## 7. Topology and device coverage


In [ ]:
topology_fields = [
    "olt_id", "pon_port", "splitter_l1", "splitter_l2", "geo_cluster"
]
topology_columns = ["ont_id", *topology_fields, "vendor", "device_model"]
topology_frame = pd.read_csv(
    TOPOLOGY, usecols=lambda name: name in topology_columns
)
required_topology = {"ont_id", *topology_fields}
missing_topology = required_topology - set(topology_frame.columns)
if missing_topology:
    raise ValueError(f"Topology fields missing: {sorted(missing_topology)}")

topology_report = pd.DataFrame({
    "field": topology_fields,
    "unique_values": [topology_frame[field].nunique(dropna=True) for field in topology_fields],
    "missing_rows": [topology_frame[field].isna().sum() for field in topology_fields],
})
display(topology_report)

device_fields = [name for name in ["vendor", "device_model"] if name in topology_frame]
if device_fields:
    display(topology_frame.groupby(device_fields, dropna=False).size().rename("ONTs").reset_index())


## 8. Compact visual audit


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
metric_quality_report.sort_values("null_rate").plot.barh(
    x="metric_id", y="null_rate", ax=axes[0], legend=False, color="#4C78A8"
)
axes[0].set_title("Null-valued rows by metric")
axes[0].set_xlabel("fraction of source rows")

entity_coverage["timestamp_gaps"].plot.hist(
    bins=25, ax=axes[1], color="#F58518", edgecolor="white"
)
axes[1].set_title("Collection gaps per ONT")
axes[1].set_xlabel("gaps longer than 1.5 × cadence")
plt.tight_layout()
plt.show()


## 9. Save the evidence tables


In [ ]:
data_quality_report = pd.DataFrame([
    {"check": "rows", "value": int(overview.loc[0, "rows"]), "status": "observed"},
    {"check": "entities", "value": int(overview.loc[0, "entities"]), "status": "observed"},
    {"check": "duplicate_entity_timestamps", "value": int(overview.loc[0, "duplicate_entity_timestamps"]), "status": "pass"},
    {"check": "median_cadence_seconds", "value": float(entity_coverage["median_cadence_seconds"].median()), "status": "pass"},
    {"check": "unreviewed_columns", "value": int(len(unreviewed)), "status": "pass"},
])

reports = {
    "data_quality_report": data_quality_report,
    "metric_quality_report": metric_quality_report,
    "entity_coverage": entity_coverage,
    "counter_audit": counter_audit,
    "topology_report": topology_report,
    "column_classification": column_classification,
}
manifest = {
    "dataset": "synthetic_pon",
    "source": str(SOURCE),
    "files_read": [str(PANEL), str(TOPOLOGY)],
    "evaluation_files_read": [],
    "rows": {name: len(frame) for name, frame in reports.items()},
}

if OUTPUT_ROOT.exists():
    previous = read_json(OUTPUT_ROOT / "audit_manifest.json")
    assert previous["dataset"] == "synthetic_pon"
    print("Using existing immutable audit output:", OUTPUT_ROOT)
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        for name, frame in reports.items():
            frame.to_parquet(output / f"{name}.parquet", index=False)
        write_json(output / "audit_manifest.json", manifest)
    print("Saved:", OUTPUT_ROOT)

connection.close()
print("PASS — every observable column was classified and no evaluation file was read")
